# Athena emulator — iceberg tables

You're a data engineer who needs ACID behavior on the lake — row-level
deletes and appends without rewriting partitions. In a handful of cells
you'll create an Iceberg table with `wr.athena.to_iceberg`, inspect the
Glue markers that route it to the dedicated catalog, append more data,
merge-delete rows by key, and join it to a plain Hive table.

Stack (`docker-compose.yml`, network `mlops_net`): `athena` (emulator) ·
`trino` (query engine) · `moto` (S3/Glue) · `jupyterlab` (this notebook).
Per ADR-0002, only Athena traffic goes to the emulator; S3/Glue calls must
still land on moto.


In [1]:
import os

# Compose sets these for in-container runs; the localhost fallbacks let the
# same notebook run against the published ports from the host.
os.environ.setdefault("AWS_ENDPOINT_URL", "http://localhost:5000")
os.environ.setdefault("AWS_ENDPOINT_URL_ATHENA", "http://localhost:5001")
os.environ.setdefault("AWS_ACCESS_KEY_ID", "test")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "test")
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")

ATHENA_URL = os.environ["AWS_ENDPOINT_URL_ATHENA"]
EDGE_URL = os.environ["AWS_ENDPOINT_URL"]

print(f"athena → {ATHENA_URL}")
print(f"s3/glue → {EDGE_URL}")

athena → http://localhost:5001
s3/glue → http://localhost:5000


In [2]:
import time
import uuid

import awswrangler as wr
import boto3
import pandas as pd
from _evidence import persist, record, render
from _helpers import delete_run_buckets, delete_run_databases

athena = boto3.client("athena")
s3 = boto3.client("s3")
glue = boto3.client("glue")

assert athena.meta.endpoint_url.rstrip("/") == ATHENA_URL, (
    athena.meta.endpoint_url
)
assert s3.meta.endpoint_url.rstrip("/") == EDGE_URL, s3.meta.endpoint_url
assert glue.meta.endpoint_url.rstrip("/") == EDGE_URL, glue.meta.endpoint_url

RUN_SUFFIX = uuid.uuid4().hex[:8]
# Every resource this run creates carries the nb06- prefix (nb06_ for Glue
# databases and tables, where '-' is invalid) so cleanup can discover
# leftovers — its own and any failed run's — instead of tracking a list.
ATHENA_RESOURCE_PREFIX = "nb06-"
DATABASE_PREFIX = "nb06_"

bucket = f"{ATHENA_RESOURCE_PREFIX}{uuid.uuid4().hex[:12]}"
database = f"{DATABASE_PREFIX}{RUN_SUFFIX}"
ice_table = f"{DATABASE_PREFIX}ice_{RUN_SUFFIX}"
hive_table = f"{DATABASE_PREFIX}hive_{RUN_SUFFIX}"
results_prefix = f"s3://{bucket}/results/"

s3.create_bucket(Bucket=bucket)
glue.create_database(DatabaseInput={"Name": database})


def glue_table(name: str) -> dict[str, object]:
    """The Glue ``Table`` member for ``name`` in this run's database."""
    return glue.get_table(DatabaseName=database, Name=name)["Table"]


print(f"bucket={bucket} database={database} suffix={RUN_SUFFIX}")

bucket=nb06-475e950f4d0d database=nb06_3ae59837 suffix=3ae59837


## Write the table

`to_iceberg` creates the table on first call — a `CREATE TABLE …
TBLPROPERTIES('table_type'='ICEBERG')` DDL — then lands the frame with a
staged `INSERT INTO … SELECT` (ADR-0013).


In [3]:
orders = pd.DataFrame(
    {"id": [1, 2, 3], "item": ["a", "b", "c"], "amount": [9.5, 1.0, 4.25]}
)

started = time.monotonic()
wr.athena.to_iceberg(
    df=orders,
    database=database,
    table=ice_table,
    temp_path=f"s3://{bucket}/iceberg-tmp-1/",
    table_location=f"s3://{bucket}/iceberg/{ice_table}/",
    s3_output=results_prefix,
)
create_ms = (time.monotonic() - started) * 1000

frame = wr.athena.read_sql_query(
    sql=f'SELECT * FROM "{ice_table}" ORDER BY "id"',
    database=database,
    ctas_approach=False,
    s3_output=results_prefix,
)
assert frame["id"].tolist() == [1, 2, 3]
assert frame["item"].tolist() == ["a", "b", "c"]

record(
    "wr.athena.to_iceberg creates + loads an Iceberg table",
    "PASS",
    f"{ice_table}: {len(frame)} row(s) read back",
    latency_ms=create_ms,
)
frame

,id,item,amount
0,1,a,9.50
1,2,b,1.00
2,3,c,4.25


## The Glue marker that routes the engine

Registration carries `table_type=ICEBERG`, `metadata_location`, and a
per-column `iceberg.field.current` — the marker that routes later
statements to the `iceberg` Trino catalog (ADR-0013).


In [4]:
registered = glue_table(ice_table)
parameters = registered["Parameters"]
storage = registered["StorageDescriptor"]

assert parameters["table_type"] == "ICEBERG"
assert parameters["metadata_location"].startswith(
    f"s3://{bucket}/iceberg/{ice_table}/"
)
assert storage["Location"].rstrip("/") == (
    f"s3://{bucket}/iceberg/{ice_table}"
)
field_markers = [
    column["Parameters"]["iceberg.field.current"]
    for column in storage["Columns"]
]
assert len(field_markers) == len(storage["Columns"]) == 3

record(
    "Glue registration carries the Iceberg routing markers",
    "PASS",
    f"table_type={parameters['table_type']}; "
    f"metadata_location=…/{parameters['metadata_location'].split('/')[-1]}; "
    f"iceberg.field.current on {len(field_markers)} column(s)",
)

Evidence(feature='Glue registration carries the Iceberg routing markers', status='PASS', detail='table_type=ICEBERG; metadata_location=…/00001-3723505a-0c5f-4cec-8b86-94e6b4c1dc55.metadata.json; iceberg.field.current on 3 column(s)', latency_ms=None)

## Append

A second `to_iceberg` skips the CREATE and runs `INSERT INTO` against the
existing table — each call stages under its own `temp_path` because
`keep_files` leaves the first call's files behind.


In [5]:
late_orders = pd.DataFrame({"id": [4], "item": ["d"], "amount": [7.0]})
wr.athena.to_iceberg(
    df=late_orders,
    database=database,
    table=ice_table,
    temp_path=f"s3://{bucket}/iceberg-tmp-2/",
    s3_output=results_prefix,
)

frame = wr.athena.read_sql_query(
    sql=f'SELECT * FROM "{ice_table}" ORDER BY "id"',
    database=database,
    ctas_approach=False,
    s3_output=results_prefix,
)
assert frame["id"].tolist() == [1, 2, 3, 4]

record(
    "wr.athena.to_iceberg appends to an existing table",
    "PASS",
    f"{len(frame)} row(s) after the second write: ids={frame['id'].tolist()}",
)

Evidence(feature='wr.athena.to_iceberg appends to an existing table', status='PASS', detail='4 row(s) after the second write: ids=[1, 2, 3, 4]', latency_ms=None)

## Delete rows by key

`delete_from_iceberg_table` stages the keys as a temp table and issues
`MERGE INTO … WHEN MATCHED THEN DELETE` — row-level deletes without
rewriting the dataset.


In [6]:
started = time.monotonic()
wr.athena.delete_from_iceberg_table(
    df=pd.DataFrame({"id": [1, 3]}),
    database=database,
    table=ice_table,
    merge_cols=["id"],
    temp_path=f"s3://{bucket}/iceberg-del-tmp/",
    s3_output=results_prefix,
)
delete_ms = (time.monotonic() - started) * 1000

frame = wr.athena.read_sql_query(
    sql=f'SELECT * FROM "{ice_table}" ORDER BY "id"',
    database=database,
    ctas_approach=False,
    s3_output=results_prefix,
)
assert frame["id"].tolist() == [2, 4]
assert frame["item"].tolist() == ["b", "d"]

record(
    "wr.athena.delete_from_iceberg_table merge-deletes rows by key",
    "PASS",
    f"deleted ids=[1, 3]; remaining={frame['id'].tolist()}",
    latency_ms=delete_ms,
)
frame

,id,item,amount
0,2,b,1.0
1,4,d,7.0


## Hive and Iceberg side by side

References route per-statement on the Glue marker: a query can join an
Iceberg table to a plain Hive table — the Iceberg reference resolves in
the `iceberg` catalog, the Hive one stays in the session catalog.


In [7]:
products = pd.DataFrame({"id": [2, 4], "label": ["bravo", "delta"]})
wr.s3.to_parquet(df=products, path=f"s3://{bucket}/hive/{hive_table}.parquet")
wr.catalog.create_parquet_table(
    database=database,
    table=hive_table,
    path=f"s3://{bucket}/hive/",
    columns_types={"id": "bigint", "label": "string"},
)

joined = wr.athena.read_sql_query(
    sql=(
        f'SELECT h."label", i."amount" FROM "{hive_table}" h, '
        f'"{ice_table}" i WHERE h."id" = i."id" ORDER BY h."id"'
    ),
    database=database,
    ctas_approach=False,
    s3_output=results_prefix,
)
assert joined["label"].tolist() == ["bravo", "delta"]
assert joined["amount"].tolist() == [1.0, 7.0]

record(
    "mixed Hive + Iceberg join routes per-reference on the Glue marker",
    "PASS",
    f"{hive_table} (hive) ⨝ {ice_table} (iceberg) → {len(joined)} row(s)",
)
joined

,label,amount
0,bravo,1.0
1,delta,7.0


## Capabilities demonstrated

Every feature above recorded a measured row. The matrix persists to
`parity/06_iceberg_tables.md`; `PARITY.md` at the project root is
regenerated from all fragments.


In [8]:
print(render())
persist("06_iceberg_tables")

| feature | status | detail | latency_ms |
|---|---|---|---|
| wr.athena.to_iceberg creates + loads an Iceberg table | PASS | nb06_ice_3ae59837: 3 row(s) read back | 3249.88 |
| Glue registration carries the Iceberg routing markers | PASS | table_type=ICEBERG; metadata_location=…/00001-3723505a-0c5f-4cec-8b86-94e6b4c1dc55.metadata.json; iceberg.field.current on 3 column(s) |  |
| wr.athena.to_iceberg appends to an existing table | PASS | 4 row(s) after the second write: ids=[1, 2, 3, 4] |  |
| wr.athena.delete_from_iceberg_table merge-deletes rows by key | PASS | deleted ids=[1, 3]; remaining=[2, 4] | 2187.8 |
| mixed Hive + Iceberg join routes per-reference on the Glue marker | PASS | nb06_hive_3ae59837 (hive) ⨝ nb06_ice_3ae59837 (iceberg) → 2 row(s) |  |



PosixPath('/home/gabriel-menezes/Documents/repos/mlops/projects/athena_emulator/parity/06_iceberg_tables.md')

## Clean up

The run's bucket and Glue database are throwaway — delete them so the
stack is left as you found it.


In [9]:
delete_run_buckets(s3, ATHENA_RESOURCE_PREFIX)
delete_run_databases(glue, DATABASE_PREFIX)
print("cleaned: nb06 buckets + databases removed")

cleaned: nb06 buckets + databases removed
